In [0]:
CREATE OR REPLACE VIEW gold_environment_monthly AS
WITH validated AS (
  SELECT
    CAST(latitude AS DOUBLE)                        AS latitude,
    CAST(longitude AS DOUBLE)                       AS longitude,
    h3_res4,
    h3_res5,
    h3_res6,
    CAST(sea_surface_temperature AS DOUBLE)         AS sst,
    CAST(chlorophyll_concentration AS DOUBLE)       AS cc,
    CAST(year AS INTEGER)                           AS year,
    CAST(month AS INTEGER)                          AS month
  FROM workspace.mcphersonsharyn_silver.environmental_conditions_silver
  WHERE
    sea_surface_temperature   IS NOT NULL
    AND sea_surface_temperature != -999
    AND sea_surface_temperature BETWEEN -2 AND 40
    AND chlorophyll_concentration IS NOT NULL
    AND chlorophyll_concentration != -999
    AND chlorophyll_concentration > 0
    AND chlorophyll_concentration < 100
    AND latitude  BETWEEN -90  AND 90
    AND longitude BETWEEN -180 AND 180
),
deduplicated AS (
  SELECT *,
    ROW_NUMBER() OVER (
      PARTITION BY h3_res4, year, month
      ORDER BY sst DESC
    ) AS rn
  FROM validated
)
SELECT
  latitude,
  longitude,
  h3_res4,
  h3_res5,
  h3_res6,
  sst,
  cc,
  year,
  month
FROM deduplicated
WHERE rn = 1;

In [0]:
SELECT
  count(*) AS total_rows,
  count(*) filter(WHERE sst IS NULL) AS null_sst,
  count(*) filter(WHERE cc IS NULL) AS null_cc,
  count(*) filter(WHERE h3_res4 IS NULL) AS null_h3,
  min(sst) AS min_sst,
  max(sst) AS max_sst,
  min(cc) AS min_cc,
  max(cc) AS max_cc,
  count(DISTINCT month) AS months_loaded
FROM gold_environment_monthly;

--new checks after adding h3  - geospacial indexing

In [0]:
SELECT count(*) FROM gold_environment_monthly;

--new checks after adding h3 - geospacial indexing.

In [0]:
describe gold_environment_monthly;

In [0]:
select * from gold_environment_monthly
limit 100;

In [0]:
select distinct month 
from gold_environment_monthly
order by month;

In [0]:
select year, month, count(*) as row_count
from mcphersonsharyn_silver.environmental_conditions_silver
group by year, month
order by year, month;